# Qwen Phase 4 AgentDojo rerun

Regenerates only the Qwen3.5-9B AgentDojo records with harness version 6. Phases 1–3, BIPIA, and InjecAgent are reused from the finished run, so nothing else is regenerated.

Harness 6 is required. It decodes injection text inside Python `repr` tool results, drops text after the first tool call, and rewrites Qwen's empty no-argument calls from `<function=name></function>` to `<function=name>{}</function>`.

The finished run is `My Drive/jspace-research/runs/jspace-qwen`. Inside it, the notebook reads `phase1/`, `phase3_implementation/`, and `phase4_implementation (1)/`. It copies those into `jspace-qwen-agentdojo-rerun` as `phase1/`, `phase3/`, and `phase4/`, and writes the new AgentDojo outputs there. The shared folder is only read.

## 1. Install the fixed branch and pinned benchmark checkouts

In [ ]:
import subprocess
from pathlib import Path

RESEARCH_REPO = 'https://github.com/ethanncyb/jspace-research.git'
RESEARCH_REVISION = 'debuging-phase-4'
BIPIA_REVISION = 'a004b69ec0dd446e0afd461d98cb5e96e120a5d0'
AGENTDOJO_REVISION = '089ed468cf3ed0322acc66b0211f26d9d90dbf60'
INJECAGENT_REVISION = 'f19c9f2c79a41046eb13c03c51a24c567a8ffa07'
REPO_ROOT = Path('/content/jspace-research')
BIPIA_CHECKOUT = Path('/content/BIPIA')
AGENTDOJO_CHECKOUT = Path('/content/agentdojo')
INJECAGENT_CHECKOUT = Path('/content/InjecAgent')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', RESEARCH_REVISION, RESEARCH_REPO, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'merge', '--ff-only', f'origin/{RESEARCH_REVISION}'], check=True)
if not BIPIA_CHECKOUT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/microsoft/BIPIA.git', str(BIPIA_CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(BIPIA_CHECKOUT), 'checkout', BIPIA_REVISION], check=True)
if not AGENTDOJO_CHECKOUT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ethz-spylab/agentdojo.git', str(AGENTDOJO_CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(AGENTDOJO_CHECKOUT), 'checkout', AGENTDOJO_REVISION], check=True)
if not INJECAGENT_CHECKOUT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/uiuc-kang-lab/InjecAgent.git', str(INJECAGENT_CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(INJECAGENT_CHECKOUT), 'checkout', INJECAGENT_REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-e', str(REPO_ROOT) + '[phase4]'], check=True)
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('AgentDojo revision:', subprocess.check_output(['git', '-C', str(AGENTDOJO_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('InjecAgent revision:', subprocess.check_output(['git', '-C', str(INJECAGENT_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
harness_source = (REPO_ROOT / 'src/jspace_research/phase4/agentdojo.py').read_text()
assert 'HARNESS_VERSION = 6' in harness_source, (
    'This checkout does not contain harness version 6. Push the Qwen empty-call fix and rerun this cell.'
)
assert 'def _fill_empty_native_tool_call' in harness_source
print('Qwen AgentDojo harness fixes are present.')

## 2. Authenticate

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import notebook_login

notebook_login()
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('OpenRouter judge credential loaded from Colab Secrets.')

## 3. Configure paths

In [ ]:
import shutil

SHARED_RUN = Path('/content/drive/MyDrive/jspace-research/runs/jspace-qwen')
RUN_ROOT = Path('/content/drive/MyDrive/jspace-research/runs/jspace-qwen-agentdojo-rerun')
DISCONNECT_WHEN_DONE = False  # True unmounts Drive and releases the runtime after Phase 4

from google.colab import drive
drive.mount('/content/drive')

PHASE1_DIR = RUN_ROOT / 'phase1'
PHASE3_DIR = RUN_ROOT / 'phase3'
PHASE4_DIR = RUN_ROOT / 'phase4'
BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'
CONFIG_PATH = REPO_ROOT / 'configs' / 'phase1_qwen3_5_9b_full.yaml'

def run_command(command, label):
    print('Running:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with exit status {return_code}; see the traceback above.')

print('Config:', CONFIG_PATH)
print('Shared run:', SHARED_RUN)
print('Run root:', RUN_ROOT)

## 4. Copy the finished run into your own folder

Runs once; rerunning the cell after the copy finishes does nothing. The pre-fix AgentDojo records are kept as `agentdojo_records_before_fix.jsonl`. The pre-fix trajectories stay in the shared folder and are not copied into the live output names.

In [ ]:
COPY_MARKER = RUN_ROOT / '.copied_from_shared_run'
OLD_AGENTDOJO = 'agentdojo_records.jsonl'

def copy_if_different(source, target):
    source, target = Path(source), Path(target)
    if target.exists() and target.stat().st_size == source.stat().st_size:
        return target
    return shutil.copy2(source, target)

SHARED_PHASES = {
    'phase1': 'phase1',
    'phase3_implementation': 'phase3',
    'phase4_implementation (1)': 'phase4',
}

if COPY_MARKER.exists():
    print('Already copied; reusing', RUN_ROOT)
else:
    assert SHARED_RUN.is_dir(), f'Add a Drive shortcut to the finished Qwen run at {SHARED_RUN}'
    for source_name, dest_name in SHARED_PHASES.items():
        print('Copying', source_name, '->', dest_name, '...')
        shutil.copytree(
            SHARED_RUN / source_name,
            RUN_ROOT / dest_name,
            dirs_exist_ok=True,
            copy_function=copy_if_different,
            ignore=shutil.ignore_patterns(OLD_AGENTDOJO, 'agentdojo_trajectories.jsonl'),
        )
    old_records = SHARED_RUN / 'phase4_implementation (1)' / OLD_AGENTDOJO
    if old_records.is_file():
        shutil.copy2(old_records, PHASE4_DIR / 'agentdojo_records_before_fix.jsonl')
    needed = ['phase1/selected_layer.json', 'phase4/bipia_records.jsonl', 'phase4/injecagent_records.jsonl']
    missing = [name for name in needed if not (RUN_ROOT / name).is_file()]
    assert not missing, 'The shared run is missing files; ask the owner for them:\n' + '\n'.join(missing)
    COPY_MARKER.write_text(f'{SHARED_RUN}\n')
    print('Copied into', RUN_ROOT)

## 5. Archive AgentDojo outputs from an older harness

Outputs from an earlier harness version are renamed with a `_harness_vN` suffix so Phase 4 regenerates AgentDojo. The finished Qwen trajectories have no harness version and are archived as version 3. Outputs from harness 6 are left to resume.

In [ ]:
import json
import re

harness_source = (REPO_ROOT / 'src/jspace_research/phase4/agentdojo.py').read_text()
HARNESS_VERSION = int(re.search(r'^HARNESS_VERSION = (\d+)', harness_source, re.MULTILINE).group(1))
assert HARNESS_VERSION >= 6, 'Rerun the install cell to pull the Qwen harness fix'
records_path = PHASE4_DIR / 'agentdojo_records.jsonl'
trajectories_path = PHASE4_DIR / 'agentdojo_trajectories.jsonl'
if trajectories_path.exists():
    with trajectories_path.open() as stream:
        first_line = stream.readline()
    previous_version = json.loads(first_line).get('harness_version', 3) if first_line.strip() else HARNESS_VERSION
    if previous_version != HARNESS_VERSION:
        for path in (records_path, trajectories_path):
            if path.exists():
                path.rename(path.with_name(f'{path.stem}_harness_v{previous_version}.jsonl'))
        print(f'Archived AgentDojo outputs from harness version {previous_version}')
    else:
        print(f'AgentDojo outputs already use harness version {HARNESS_VERSION}; Phase 4 will resume them')
else:
    print(f'No AgentDojo trajectories yet; Phase 4 will generate with harness version {HARNESS_VERSION}')

## 6. Verify the reused inputs

The config identity must match the finished run (`phase4_config_sha256` `f18c79e33c85…`). That is what lets Phase 4 reuse the BIPIA and InjecAgent records instead of regenerating them.

In [ ]:
import hashlib
import json

from jspace_research.phase4.config import load_config

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

required = [
    PHASE1_DIR / 'selected_layer.json',
    PHASE1_DIR / 'provenance.json',
    PHASE3_DIR / 'mean_detector.pt',
    PHASE3_DIR / 'logistic_detector.pt',
    PHASE4_DIR / 'provenance.json',
    PHASE4_DIR / 'bipia_test_manifest.jsonl',
    PHASE4_DIR / 'bipia_records.jsonl',
    PHASE4_DIR / 'bipia_judgments.jsonl',
    PHASE4_DIR / 'injecagent_records.jsonl',
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, 'Missing files; without the records Phase 4 would regenerate BIPIA/InjecAgent:\n' + '\n'.join(missing)

new_records = PHASE4_DIR / 'agentdojo_records.jsonl'
assert not new_records.exists() or (PHASE4_DIR / 'agentdojo_trajectories.jsonl').exists(), (
    'agentdojo_records.jsonl has no trajectories file, so it is the pre-fix run; delete it before Phase 4'
)

provenance = json.loads((PHASE4_DIR / 'provenance.json').read_text())
config = load_config(
    CONFIG_PATH,
    phase1_selected_path=PHASE1_DIR / 'selected_layer.json',
    phase3_dir=PHASE3_DIR,
    bipia_root=BIPIA_ROOT,
    agentdojo_root=AGENTDOJO_CHECKOUT,
    injecagent_root=INJECAGENT_CHECKOUT,
    output_dir=PHASE4_DIR,
)
assert config.identity_hash() == provenance['phase4_config_sha256'], (
    'phase1_qwen3_5_9b_full.yaml does not match the finished Qwen run; '
    'BIPIA and InjecAgent would not be reused'
)
assert config.phase1.model.id == 'Qwen/Qwen3.5-9B'
for name in ('mean', 'logistic'):
    assert sha256(PHASE3_DIR / f'{name}_detector.pt') == provenance['detectors'][f'{name}_sha256'], (
        f'{name} detector hash differs from Phase 4 provenance'
    )
for name in ('bipia_records', 'injecagent_records', 'agentdojo_records'):
    path = PHASE4_DIR / f'{name}.jsonl'
    count = sum(1 for _ in path.open()) if path.exists() else 0
    print(f'{name}: {count} rows')
print('Config, detectors, and reused records match the finished Qwen run.')

## 7. Verify the GPU runtime

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before running the experiment.'
print('GPU:', torch.cuda.get_device_name(0))

## 8. Run Phase 4

BIPIA and InjecAgent load from their saved records; only AgentDojo is generated. The run resumes from `agentdojo_records.jsonl` if interrupted.

In [ ]:
import os

import pandas as pd
from IPython.display import Image, display

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

phase4_base = [
    'jspace-phase4',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--phase3', str(PHASE3_DIR),
    '--bipia-root', str(BIPIA_ROOT),
    '--agentdojo-root', str(AGENTDOJO_CHECKOUT),
    '--injecagent-root', str(INJECAGENT_CHECKOUT),
    '--output-dir', str(PHASE4_DIR),
]
try:
    run_command([*phase4_base, '--stage', 'generate'], 'Phase 4 generation')
    run_command([*phase4_base, '--stage', 'analyze'], 'Phase 4 analysis')
    display(pd.read_csv(PHASE4_DIR / 'phase4_metrics.csv'))
    display(Image(filename=str(PHASE4_DIR / 'phase4_detector_transfer.png')))
finally:
    if DISCONNECT_WHEN_DONE:
        from google.colab import runtime
        drive.flush_and_unmount()
        runtime.unassign()

## 9. Quick AgentDojo sanity check

A usable rerun has harness version 6 on every trajectory, no attack success marked unseen, and no empty `<function=name></function>` left in the parsed text.

In [ ]:
records = pd.read_json(PHASE4_DIR / 'agentdojo_records.jsonl', lines=True)
attacks = records[records['condition'] == 'attack']
print('Records:', len(records), '| attacks:', len(attacks), '| controls:', len(records) - len(attacks))
print('Unscored episodes:', int(records['mean_score'].isna().sum()))
exposed = attacks['injection_exposed'].astype(bool)
success = attacks['native_attack_success'].fillna(False).astype(bool)
print('Attack successes without exposure:', int((success & ~exposed).sum()))
display(
    attacks.assign(exposed=exposed, success=success)
    .groupby('suite')[['exposed', 'success']]
    .mean()
    .round(3)
)

trajectories = {}
with (PHASE4_DIR / 'agentdojo_trajectories.jsonl').open() as stream:
    for line in stream:
        row = json.loads(line)
        trajectories[row['case_id']] = row
versions = {row.get('harness_version') for row in trajectories.values()}
print('Harness versions:', versions)
empty_calls = sum(
    1
    for row in trajectories.values()
    for step in row['steps']
    if re.search(r'<function\s*=\s*[A-Za-z_][A-Za-z0-9_]*\s*>\s*</function>', step.get('parsed_text') or '')
)
print('Parsed steps that still have an empty function body:', empty_calls)
never_parsed = sum(
    1 for row in trajectories.values() if not any(step.get('parsed_tool_calls') for step in row['steps'])
)
print('Episodes that never executed a tool call:', never_parsed, 'of', len(trajectories))
capped = sum(1 for row in trajectories.values() if any(step.get('generated_tokens') == 512 for step in row['steps']))
print('Episodes with a step at the 512-token cap:', capped)